# YOLO26n exploration

Exploring Ultralytics YOLO26 nano on CPU. Released Jan 2026 — end-to-end (no NMS), ~39 ms ONNX CPU inference, 40.9 mAP on COCO val2017.

This notebook is the upstream context for the IRON design in the parent directory: it shows how to load YOLO26n, run inference, and validate against COCO. The classification variant (`yolo26n-cls.pt`) fine-tuned on a person/no-person dataset is the FP32 source that `quark_quantization.ipynb` then quantizes to INT8 for NPU deployment.


In [ ]:
%pip install -q -U ultralytics

## Load YOLO26n

In [ ]:
from ultralytics import YOLO
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

model = YOLO("yolo26n.pt")  # auto-downloads weights on first call
model.info()

## Inference on a sample image

In [ ]:
from PIL import Image as PILImage

results = model("https://ultralytics.com/images/bus.jpg")
r = results[0]

print(f"{len(r.boxes)} detections:")
for cls, conf in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()):
    print(f"  {r.names[int(cls)]:<15} conf={conf:.3f}")

# .plot() returns a BGR numpy array with boxes drawn; convert to RGB for display.
PILImage.fromarray(r.plot()[..., ::-1])

## COCO datasets

Ultralytics auto-downloads datasets into `settings["datasets_dir"]` (default: `~/datasets/`). Three built-in YAMLs cover the common cases:

| YAML | Images | Approx. size | Use |
|---|---|---|---|
| `coco8.yaml` | 8 | <1 MB | smoke test |
| `coco128.yaml` | 128 | ~7 MB | quick val sanity check |
| `coco.yaml` | val2017: 5k / train2017: 118k | val ~1 GB, train ~18 GB | real benchmarking + training |

`.val(data="coco.yaml")` only pulls val2017. `.train(data="coco.yaml")` also pulls train2017.

In [ ]:
from ultralytics import settings

print("datasets dir:", settings["datasets_dir"])
# To change: settings.update({"datasets_dir": "/path/you/prefer"})

### Quick smoke-test validation on coco128

In [ ]:
metrics = model.val(data="coco128.yaml")
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)

### Full val2017 benchmark (~1 GB download)

In [ ]:
metrics_full = model.val(data="coco.yaml")
print("mAP50-95:", metrics_full.box.map)

### Training on full COCO (~18 GB train2017 download)

GPU strongly recommended. CPU training on full COCO is impractical (days per epoch).

In [ ]:
#### For tiny test case with coco128 (small-dataset recipe)
# YOLO26 default hyperparameters are tuned for the full COCO recipe (batch=128, lr0=0.0054).
# On small datasets like coco128 (128 images), defaults destroy the pretrained weights —
# loss flatlines and mAP collapses to ~0. The Ultralytics YOLO26 training recipe doc
# (https://docs.ultralytics.com/guides/yolo26-training-recipe) prescribes these settings
# for datasets <1,000 images:
#   - lr0=0.001            (10× lower than default 0.01)
#   - freeze=10            (freeze backbone, fine-tune the head only)
#   - mosaic=0.5           (reduce heavy augmentation)
#   - mixup=0.0            (off for small datasets)
#   - copy_paste=0.0       (off for small datasets)
#   - epochs=50 patience=20
# With these, fine-tuning on coco128 lifts mAP50-95 from 0.476 (pretrained) to ~0.53.

# IMPORTANT: reload fresh pretrained weights at the start of every training run.
# model.train() mutates `model` in place, so re-running this cell after a failed
# training would start from the previously-corrupted weights, not yolo26n.pt.
model = YOLO("yolo26n.pt")

# device: "cpu", "cuda", or "mps" (Apple Silicon) — pick whatever your host has.
results = model.train(
    data="coco128.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    device="cpu",
    lr0=0.001,
    freeze=10,
    mosaic=0.5,
    mixup=0.0,
    copy_paste=0.0,
    patience=20,
    amp=False,
)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

save_dir = Path(results.save_dir)

# Per-epoch metrics table (loss, mAP, etc.)
df = pd.read_csv(save_dir / "results.csv")
df.tail()

# The headline plot: losses + mAP curves over epochs
display(Image(save_dir / "results.png"))

# Confusion matrix + PR curve.
# YOLO26 prefixes detection-curve files with "Box" (BoxPR_curve.png, BoxF1_curve.png, ...)
# rather than the plain "PR_curve.png" name used in older YOLO versions.
display(Image(save_dir / "confusion_matrix.png"))
display(Image(save_dir / "BoxPR_curve.png"))

# Side-by-side: ground-truth labels vs model predictions on a val batch
display(Image(save_dir / "val_batch0_labels.jpg"))
display(Image(save_dir / "val_batch0_pred.jpg"))

## CPU latency benchmark

In [ ]:
import time
import urllib.request
from pathlib import Path
import numpy as np

img_path = Path("bus.jpg")
if not img_path.exists():
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", img_path)

_ = model(str(img_path), verbose=False)  # warm up

n = 20
times = []
for _ in range(n):
    t0 = time.perf_counter()
    _ = model(str(img_path), verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

print(
    f"CPU latency over {n} runs: mean {np.mean(times):.1f}ms | p50 {np.percentile(times,50):.1f}ms | p95 {np.percentile(times,95):.1f}ms"
)

## Next steps

- **Other size variants:** `yolo26s.pt`, `yolo26m.pt`, `yolo26l.pt`, `yolo26x.pt` for accuracy/speed tradeoffs
- **ONNX export:** `model.export(format="onnx")` — the 38.9 ms CPU figure Ultralytics quotes is via ONNX Runtime, not PyTorch
- **Other tasks:** `yolo26n-seg.pt` (segmentation), `yolo26n-pose.pt` (pose), `yolo26n-obb.pt` (oriented boxes), `yolo26n-cls.pt` (classification)
- **Predict on your data:** pass a file path, folder, glob, video, or webcam index to `model(...)`